# Prepare data for data publishing for JAMES (at submission stage)

What we need:
- Monthly mean online, monthly offline and daily offline raw post-processed budget output for 2019 (Figs. 1, 3, 4).
- Monthly mean online and monthly offline post-processed budget climatologies (Figs. 1, 3-10).
- Monthly mean online and monthly offline post-processed budget output for 2011 and 2017 (Tasman Sea and WA events - spatial plots).
- ocean_grid.nc
(These were all just copied from the post-processed diagnostics files, totalling about 6 or 7GB).

- Single-point output for Fig. 2.
- Cutdown (to Australia) daily budget, MLD, MLT and MHW threshold data for MHW cases:
  - November 2017- Feb 2018
  - Feb - May 2011

In [ ]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
from tqdm import tqdm
import glob
import sys, os

In [ ]:
base = '/g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/'
outname = 'ACCESS-OM2-025_ERA5_'
pp_diags_folder = base + 'post_processed_diags/'
base_out = '/g/data/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/data_publishing/'

# Region to load:
# reg = [100-360, 140-360, -45, -10] # WA - for 2011 WA event
# reg_name = 'Western_Australia'
# reg = [135-360,175-360, -60, -20] # SE Aus (Kajtar et al. 2022)
reg = [100-360, 175-360, -60, -10] # All Australia
reg_name = 'Australia'

clim_str = 'output336-365' # 336-365 = 1989-2018
clim_label = '1989-2018'

In [ ]:
# MHW climatology and thresholds:
mhw_clim = xr.open_dataset(pp_diags_folder + 'om2_025_MLT_clim.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
mhw_thresh = xr.open_dataset(pp_diags_folder + 'om2_025_MLT_thresh.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

mhw_clim.to_netcdf(base_out + 'MHW_daily_climatology_' + reg_name + '.nc')
mhw_thresh.to_netcdf(base_out + 'MHW_daily_threshold_' + reg_name + '.nc')

In [ ]:
# Climatology of standard variables:
ds_clim = xr.open_dataset(pp_diags_folder + 'ocean_month_' + clim_str + '.clim.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_clim = ds_clim.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_clim.time.values]})
ds_clim.to_netcdf(base_out + 'ocean_daily_clim_' + reg_name + '.nc')

In [ ]:
# Choose year to consider:

# WA event:
output = 358
times = slice('2011-01-01','2011-06-01')

# Tasman Sea event:
#output = 364 
#times = slice('2017-11-01','2017-12-31')
#output = 365
#times = slice('2018-01-01','2018-03-01')

base2 = base + 'output%03d/ocean/' % output

In [ ]:
# Extract variables for chosen time period:
mlt_budget_stavg_daily = xr.open_dataset(pp_diags_folder + 'mlt_budget_online_stavg/mlt_budget_stavg_daily_online_output%03d.nc' % output).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).sel(time=times).load()
mlt_budget_stavg_daily.to_netcdf(base_out + 'mlt_budget_stavg_daily_online_output%03d_' % output + reg_name + '.nc')

In [ ]:
# Daily standard variables:
ds_day = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day = ds_day.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day.time.values]})
ds_day.average_DT.data = ds_day.average_DT*np.timedelta64(1,'D')
ds_day = ds_day.sel(time=times)[['temp_in_mld','mld']].load()

# Save to file:
ds_day.to_netcdf(base_out + 'ocean_daily_output%03d_' % output + reg_name + '.nc')